# 1.ライブラリ・環境準備

In [ ]:
# インストール（1度だけ。環境によってはスキップOK）
!pip install -U keras-nlp keras-core

# 環境変数: TensorFlowバックエンド指定
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

# ライブラリまとめてimport
import numpy as np
import pandas as pd
import keras_nlp
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import re, string, html

# 2.データ読み込み

In [ ]:
# Kaggleデータパス
INPUT_DIR = '/kaggle/input/nlp-getting-started'
df_train = pd.read_csv(f"{INPUT_DIR}/train.csv")
df_test  = pd.read_csv(f"{INPUT_DIR}/test.csv")

# 3.前処理（テキストクリーニング）

In [ ]:
def clean_text(text):
    text = str(text)
    text = html.unescape(text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_train['cleaned_text'] = df_train['text'].apply(clean_text)
df_test['cleaned_text']  = df_test['text'].apply(clean_text)

# 4.データ分割

In [ ]:
VAL_SPLIT = 0.2

X_train, X_val, y_train, y_val = train_test_split(
    df_train['cleaned_text'], df_train['target'], 
    test_size=VAL_SPLIT, random_state=42
)
X_test = df_test['cleaned_text']

# 5.KerasNLPのRobertaモデル構築

In [ ]:
preset = "roberta_large_en" 

# 前処理レイヤ
preprocessor = keras_nlp.models.RobertaPreprocessor.from_preset(preset, sequence_length=160)

# 分類器本体
classifier = keras_nlp.models.RobertaClassifier.from_preset(
    preset,
    preprocessor=preprocessor,
    num_classes=2,
    dropout=0.3
)

# 6.コンパイル & 学習

In [ ]:
classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

history = classifier.fit(
    X_train, y_train,
    batch_size=20, epochs=10,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping]
)

# 7.評価（学習/検証セット）

In [ ]:
def eval_print(y_true, y_pred, tag=""):
    y_pred_label = np.argmax(y_pred, axis=1)
    print(f"\n[{tag}] Confusion Matrix")
    print(confusion_matrix(y_true, y_pred_label))
    print(classification_report(y_true, y_pred_label, digits=4))

y_pred_train = classifier.predict(X_train)
y_pred_val   = classifier.predict(X_val)

eval_print(y_train, y_pred_train, "Train")
eval_print(y_val, y_pred_val, "Validation")

# 8.テスト予測＆提出ファイル

In [ ]:
sample_submission = pd.read_csv(f"{INPUT_DIR}/sample_submission.csv")
sample_submission["target"] = np.argmax(classifier.predict(X_test), axis=1)
sample_submission.to_csv("/kaggle/working/submission.csv", index=False)